[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://drive.google.com/file/d/1OfkzFl9JHnDZJXuUQ6ztj7xR9WGry9Ut/view?usp=drive_link)

# RAG Evaluation – Partial Dataset

This notebook demonstrates how to evaluate RAG when you have questions and retrieved contexts but no model responses yet. Floeval generates responses from the question and contexts, then scores them.

**Objectives**
- Install Floeval and configure credentials
- Load a partial RAG dataset from a JSON file you provide
- Set `dataset_generator_model` for response generation
- Run evaluation and inspect results

## 1. Installation

Install Floeval before running this notebook.

In [ ]:
%pip install floeval>=0.2.0b1

## 2. Configuration Constants

Set the following constants before running. Replace placeholder values with your API credentials and model identifiers.

**Provider flexibility:** You can use any OpenAI-compatible provider (OpenAI, Azure OpenAI, Anthropic, local models, etc.) — set the appropriate `base_url` and model names for your provider.

**Using FloTorch:** If you want to use FloTorch keys and gateway, obtain credentials from the [FloTorch Console](https://docs.flotorch.cloud/introduction/).

In [ ]:
import getpass
# LLM and API configuration (OpenAI)

OPENAI_BASE_URL = "https://api.openai.com/v1"
OPENAI_API_KEY = getpass.getpass("Enter your API key: ")
OPENAI_CHAT_MODEL = "gpt-4o-mini"
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"

## 3. Imports

Import evaluation components and the LLM configuration schema.

In [ ]:
from pathlib import Path

from floeval import Evaluation, DatasetLoader
from floeval.config.schemas.io.llm import OpenAIProviderConfig

## 4. Configure the LLM

The LLM configuration is used for both response generation and evaluation.

In [ ]:
llm_config = OpenAIProviderConfig(
    base_url=OPENAI_BASE_URL,
    api_key=OPENAI_API_KEY,
    chat_model=OPENAI_CHAT_MODEL,
    embedding_model=OPENAI_EMBEDDING_MODEL,
)

## 5. Load the Partial RAG Dataset

Minimal JSON shape (omit `llm_response`; Floeval generates answers):

```json
{
  "samples": [ { "user_input": "...", "contexts": ["..."] } ]
}
```

**Example file**  
<a href="../datasets/rag_evaluation/sample_rag_partial_dataset.json" download="sample_rag_partial_dataset.json">sample_rag_partial_dataset.json</a>

Provide the partial dataset JSON path (or upload in Colab), then load it with `DatasetLoader`.


In [ ]:
try:
    from google.colab import files
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

if _IN_COLAB:
    print("Upload your dataset JSON file:")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded.")
    dataset_path = Path(next(iter(uploaded.keys())))
else:
    dataset_path = Path(input("Enter path to dataset JSON file: ").strip().strip('"')).expanduser()


### Resolve Dataset Path

Provide `dataset_path` via file upload in Colab or local JSON path input in Jupyter.


In [ ]:
partial_dataset = DatasetLoader.from_file(dataset_path, partial_dataset=True)
print(f"Partial RAG dataset loaded from {dataset_path}: {len(partial_dataset.samples)} samples")


### Load Partial RAG Dataset

Load and validate the partial RAG dataset with `DatasetLoader.from_file(..., partial_dataset=True)`.


## 6. Create and Run the Evaluation

Set `dataset_generator_model` to generate responses from `user_input` and `contexts`, then run async evaluation.

In [ ]:
evaluation = Evaluation(
    dataset=partial_dataset,
    llm_config=llm_config,
    metrics=["answer_relevancy", "faithfulness"],
    default_provider="ragas",
    dataset_generator_model=OPENAI_CHAT_MODEL,
)


### Build Evaluation Object

Configure `Evaluation` with response generation and RAG metric settings.


In [ ]:
results = await evaluation.arun()
print("Aggregate scores:", results.aggregate_scores)


### Run Evaluation (Async)

Execute `await evaluation.arun()` to compute and print aggregate metric scores.


## 7. Inspect Generated Responses

Each sample result includes the generated `llm_response` and the metric scores.

### Inspect per-sample results

Iterates `results.sample_results` to print each question snippet and metric scores.


In [ ]:
for i, sr in enumerate(results.sample_results, start=1):
    print(f"Sample {i}: {sr['user_input']}")
    print(f"  Generated: {sr.get('llm_response', '')[:100]}...")
    for key, data in sr.get("metrics", {}).items():
        print(f"  {key}: {data.get('score')}")

## Summary

This notebook demonstrated the evaluation of RAG pipelines with questions and retrieved contexts but no pre-generated responses.

The key components included:

1. **Partial RAG Dataset Loading**: A dataset with `user_input` and `contexts` was loaded from JSON using `DatasetLoader.from_file(..., partial_dataset=True)`.
2. **LLM Configuration**: The OpenAI-compatible provider was configured for both generation and evaluation.
3. **Response Generation**: The `dataset_generator_model` parameter enabled Floeval to generate responses at runtime from question and context.
4. **Evaluation Execution**: The `answer_relevancy` and `faithfulness` metrics were run on the generated responses.
5. **Results Inspection**: Generated text and per-sample scores were accessed through `results.sample_results`.

This example showcases the workflow for evaluating RAG when retrieval results exist but answers are not yet generated.